# 03 · Crowd → Image  (Stage 3 — the RQ1 aggregation study)

**Crowd-Driven Visual Generation.** Turn a **crowd of words/emojis** into one coherent image, and compare **aggregation strategies** (RQ1).

```
crowd → CLIP text embeddings → AGGREGATE (mean / centroid) → +modality-gap → one vector
      → guided DDIM (frozen diffusion) → latent → VAE decode → image
```

**Modality-gap correction:** the diffusion model was trained on CLIP *image* embeddings, but a crowd is CLIP *text* — those clouds don't coincide, so raw text conditions are out-of-distribution. We measure the offset once and translate text conditions toward the image manifold. Inference only, no training.

> Runtime → **GPU (T4)**. Needs `vae.pt` + `diffusion.pt` (Drive or HF).

## 1. Clone the repo & enter the project

In [ ]:
!git clone --branch feature/crowd-driven-visual-generation https://github.com/nishant-kumar109/gen-ai-IISc.git
%cd gen-ai-IISc/projects/crowd-driven-visual-generation

In [ ]:
!pip install -q datasets open-clip-torch matplotlib   # dataset + CLIP + figures
import torch; print('cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Credentials & locate the checkpoints
Mounts Drive and finds `vae.pt` + `diffusion.pt` (Drive first, else pulled from HF).

In [ ]:
import os, getpass
from google.colab import drive; drive.mount('/content/drive')

hf_token = getpass.getpass("HF token (Enter to skip if checkpoints are on Drive): ").strip()
if hf_token:
    from huggingface_hub import login; login(token=hf_token)

def locate(drive_path, repo, fname):
    if os.path.exists(drive_path):
        return drive_path
    from huggingface_hub import hf_hub_download, whoami
    return hf_hub_download(f"{whoami()['name']}/{repo}", fname)

VAE_CKPT  = locate('/content/drive/MyDrive/crowdgen/vae/vae.pt', 'crowdgen-vae', 'vae.pt')
DIFF_CKPT = locate('/content/drive/MyDrive/crowdgen/diffusion/diffusion.pt', 'crowdgen-diffusion', 'diffusion.pt')
OUT = '/content/drive/MyDrive/crowdgen/figures'; os.makedirs(OUT, exist_ok=True)
GAP_FILE = '/content/drive/MyDrive/crowdgen/diffusion/gap.pt'
print('VAE :', VAE_CKPT); print('DIFF:', DIFF_CKPT)

## 3. Measure the modality gap  *(once, ~1–2 min)*
Computes `gap = mean(image embeddings) − mean(text embeddings)` over ~2000 paintings + the crowd vocabulary. A large `‖gap‖` confirms the image/text clouds are far apart — i.e. why raw text conditioning was weak.

In [ ]:
!python compute_gap.py --diffusion {DIFF_CKPT} --dataset huggan/wikiart \
    --limit 2000 --out {GAP_FILE}

## 4. Themes study — before vs after gap correction
Same crowds, sampled twice: raw text conditions vs gap-corrected. The corrected grid should show **much stronger theme control** (e.g. `night` goes dark, `love` goes warm).

In [ ]:
common = ('--vae {v} --diffusion {d} --mode themes --aggregators mean,centroid '
          '--n 300 --diversity 0.2 --guidance 3.0 --steps 50').format(v=VAE_CKPT, d=DIFF_CKPT)
!python sample_crowd.py {common} --out {OUT}/themes_nogap.png
!python sample_crowd.py {common} --gap-file {GAP_FILE} --out {OUT}/themes_gap.png
from IPython.display import Image, display
print('WITHOUT gap correction (raw text conditions):'); display(Image(f'{OUT}/themes_nogap.png'))
print('WITH gap correction:');                          display(Image(f'{OUT}/themes_gap.png'))

## 5. Diversity study — the RQ1 punchline  *(raw conditions)*
Same theme, rising off-theme noise. As the crowd gets noisier, **mean-pool** should drift toward mush (it averages everything), while **cluster-centroid** should stay coherent (locks onto the dominant theme, ignores scattered noise). Using raw conditions — gap correction was found to wash themes out.

In [ ]:
!python sample_crowd.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} \
    --mode diversity --theme paradise --aggregators mean,centroid \
    --diversities 0.1,0.4,0.7 --n 300 --guidance 3.0 --steps 50 \
    --out {OUT}/crowd_diversity.png
from IPython.display import Image; Image(f'{OUT}/crowd_diversity.png')

## 6. Guidance sweep on raw conditions  *(the cheap fix to try first)*
Gap correction (§4) hurt — it collapsed themes to a washed-out average. But raw text conditions *do* differentiate themes, just weakly. Higher guidance `w` amplifies that signal. If a higher `w` sharpens theme control, we may not need a retrain at all.

In [ ]:
# RAW text conditions (gap correction washed themes out — see §4), sweep guidance strength.
# Higher w pushes the model harder toward the (weak but theme-differentiated) crowd signal.
# NOTE: build the command as an f-string so the loop variable w is interpolated by Python
# (IPython's {..} expansion in `!` doesn't reach loop variables reliably).
from IPython.display import Image, display
for w in [3.0, 5.0, 7.0, 10.0]:
    cmd = (f'python sample_crowd.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} '
           f'--mode themes --aggregators mean,centroid --n 300 --diversity 0.2 '
           f'--guidance {w} --steps 50 --out {OUT}/sweep_w{w}.png')
    !{cmd}
for w in [3.0, 5.0, 7.0, 10.0]:
    print(f'guidance w={w} (raw, no gap correction)'); display(Image(f'{OUT}/sweep_w{w}.png'))

## 7. Quantify RQ1 — numbers behind the visual story
Scores each aggregator with CLIP, per diversity (averaged over themes): **theme_fidelity** (is the image on-theme?) and **consistency** (does the aggregator map noisy crowds to a stable target?). The plots show how each metric holds up as crowd noise rises — the quantitative RQ1 result. ~3–5 min on T4.

In [ ]:
!python evaluate.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} \
    --repeats 8 --n 300 --guidance 3.0 --steps 50 --out {OUT}/rq1
from IPython.display import Image, display
display(Image(f'{OUT}/rq1/rq1_fidelity.png')); display(Image(f'{OUT}/rq1/rq1_consistency.png'))

## Next — evaluation & the report
Figures under `MyDrive/crowdgen/figures/`. If gap correction sharpened theme control, the crowd→image thesis holds. What follows:
- **Quantify RQ1**: CLIP theme-fidelity + coherence per aggregator × diversity (numbers behind the visual story).
- **Learnable aggregators**: train DeepSets / attention-pooling and add to the comparison.
- **Fold the gap-mean into the diffusion checkpoint** on the next (texture) retrain, so correction is built in.
- Optional: stronger diffusion run for crisper final art.